# GGA Module MVP — Incident Typologies from Crash Narratives

**MVP goal:** prove the whole typology loop end to end on a small stratified
sample (~400 narratives, embedding in a few minutes), before committing to the
full-corpus run. Pipeline: mask outcome words → embed narratives (GeoAI text
model) → cluster → name clusters from their own words → check whether injury
rates differ by cluster. Everything runs in this one notebook inside ArcGIS
Pro 3.7 — no R, no GPU.

**Success looks like:** clusters whose top terms read as coherent incident
mechanisms (turning vehicles, doors, surface hazards...) and an injury-rate
table where the rates differ. Then scale up (last cell).

## How to run (Pro notebooks can jam — read first)

- **Run cells one at a time, top to bottom. Never "Run All".**
- Heavy cells print a start timestamp + expected duration immediately, save
  their result to disk, and record themselves in `progress.json`. Re-running a
  finished cell skips itself.
- **If a cell hangs** far past its expected duration: geoprocessing tools often
  cannot be interrupted from the notebook. Close ArcGIS Pro (Task Manager →
  end ArcGISPro.exe if needed), reopen, run **cell 0 only**, read the printed
  checklist, continue from the first unchecked step. You lose at most the one
  step that was running.
- Keep layers **out of the map** until the final step — open layers hold
  schema locks that make re-runs hang.


## 0 — Config and progress tracker

Edit the four paths. `SAMPLE_N = 400` is the MVP knob — the sample is
stratified on injury so the small run still has enough injured cases to
validate against. Safe to re-run any time; prints the checklist.

In [ ]:
# ========================== EDIT THESE FOUR PATHS ==========================
HOME      = r"E:\GGA\Embeddings\typology_mvp"                 # working folder (created if missing)
CANADA    = r"E:\GGA\data\20240425Canada.xlsx"                # BikeMaps Canada export
US        = r"E:\GGA\data\20240425United States.xlsx"         # BikeMaps US export
TEXT_DLPK = r"E:\GGA\Embeddings\TextEncoder.dlpk"             # from the smoke test; auto-downloaded in step 3 if missing
# ===========================================================================
SAMPLE_N  = 400     # MVP sample size; set to None for the full corpus (new HOME folder then!)

import os, re, json, time, random
import arcpy
import numpy as np
import pandas as pd

SEED = 1208
random.seed(SEED); np.random.seed(SEED)

os.makedirs(HOME, exist_ok=True)
GDB      = os.path.join(HOME, "typology.gdb")
PROGRESS = os.path.join(HOME, "progress.json")
STEPS    = ["1_prep", "2_points", "3_model", "4_embed", "5_extract", "6_cluster", "7_validate"]

def _load():
    try:
        with open(PROGRESS) as f:
            return json.load(f)
    except Exception:
        return {}

def done(step):
    return _load().get(step, {}).get("done", False)

def mark(step, **info):
    p = _load()
    p[step] = {"done": True, "at": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
    with open(PROGRESS, "w") as f:
        json.dump(p, f, indent=1)
    print(f"[done] {step}", flush=True)

def begin(step, expected):
    print(f"[start] {step} at {time.strftime('%H:%M:%S')} — expected {expected}", flush=True)
    return time.time()

def status():
    p = _load()
    print("Progress:", flush=True)
    for s in STEPS:
        box = "[x]" if p.get(s, {}).get("done") else "[ ]"
        print(f"  {box} {s}", flush=True)

if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(HOME, "typology.gdb")
arcpy.env.workspace = GDB
arcpy.env.overwriteOutput = True

import sklearn
print("ArcGIS Pro", arcpy.GetInstallInfo()["Version"], "| license:", arcpy.ProductInfo(), flush=True)
print("sklearn", sklearn.__version__, "| sample:", SAMPLE_N or "FULL CORPUS", flush=True)
HAS_HDBSCAN = tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 3)
print("HDBSCAN in sklearn:", HAS_HDBSCAN, "(KMeans fallback otherwise)", flush=True)
status()

## 1 — Load, filter, mask, sample (~30 s)

Personally-involved incidents after 2016-11-30 with a ≥20-character narrative;
`injured` = any injury vs. none.

**Why mask?** 30% of injured riders' narratives contain outcome words vs 6% of
uninjured — P(injured | outcome word) = 51% against a 17% base rate. Raw-text
clusters would include an "injury stories" cluster, making the injury-rate
validation circular. Masking deletes every *sentence* with outcome vocabulary,
so clusters can only form around circumstances.

In [ ]:
STEP = "1_prep"
PREP_CSV = os.path.join(HOME, "narratives.csv")

if done(STEP) and os.path.exists(PREP_CSV):
    narr = pd.read_csv(PREP_CSV)
    print(f"skipped — already done ({len(narr)} rows)", flush=True)
else:
    t0 = begin(STEP, "~30 s")
    frames = []
    for country, path in [("CA", CANADA), ("US", US)]:
        t = pd.read_excel(path)
        t["country"] = country
        frames.append(t)
        print(f"  read {country}: {len(t)} reports", flush=True)
    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    d = df[(df["date"] > "2016-11-30")
           & (df["personal_involvement"] == "Yes")
           & df["ebike"].isin(["Yes", "No"])].copy()
    d = d[d["details"].notna()
          & (d["details"].astype(str).str.len() >= 20)
          & d["x"].notna() & d["y"].notna()]
    d["injured"] = ((~d["injury"].isin(["No injury", "Unknown"]))
                    & d["injury"].notna()).astype(int)

    LEAK = re.compile(
        r"\b(injur\w*|hospital\w*|doctor|ambulance|paramedic|brok\w*|fractur\w*"
        r"|pain\w*|bruis\w*|stitch\w*|concuss\w*|emergency|x-?ray|sprain\w*"
        r"|bleed\w*|blood|scrape\w*|road ?rash|wound\w*|surgery|clinic|e\.?r\.?)\b",
        re.I)

    def mask_text(s):
        sentences = re.split(r"(?<=[.!?])\s+", str(s))
        kept = [x for x in sentences if not LEAK.search(x)]
        return " ".join(kept).strip() or "no description"

    d["details_masked"] = d["details"].astype(str).apply(mask_text)
    d = d[d["details_masked"] != "no description"]   # MVP: keep only rows with text left after masking
    print(f"  {len(d)} narratives survive masking", flush=True)

    if SAMPLE_N and len(d) > SAMPLE_N:
        total = len(d)
        parts = [g.sample(min(int(round(SAMPLE_N * len(g) / total)), len(g)), random_state=SEED)
                 for _, g in d.groupby("injured")]
        d = pd.concat(parts)
        print(f"  stratified sample: {len(d)} rows", flush=True)

    narr = d[["pk", "country", "x", "y", "ebike", "injured",
              "details", "details_masked"]].reset_index(drop=True)
    narr.to_csv(PREP_CSV, index=False, encoding="utf-8-sig")
    print(f"{len(narr)} narratives | injured {narr.injured.mean():.1%} | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, rows=len(narr))

## 2 — Points + model (~10 s, plus a one-time model download)

In [ ]:
STEP = "2_points"
FC = os.path.join(GDB, "narratives")

if done(STEP) and arcpy.Exists(FC):
    print("2_points skipped — already done", flush=True)
else:
    t0 = begin(STEP, "~10 s")
    arcpy.management.XYTableToPoint(PREP_CSV, FC, "x", "y",
                                    coordinate_system=arcpy.SpatialReference(4326))
    n = int(arcpy.management.GetCount(FC)[0])
    print(f"{n} point features | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, n=n)

STEP = "3_model"
if os.path.exists(TEXT_DLPK):
    print("model on disk:", TEXT_DLPK, flush=True)
    if not done(STEP):
        mark(STEP, path=TEXT_DLPK)
else:
    t0 = begin(STEP, "2-10 min (one-time download)")
    from arcgis.gis import GIS
    gis = GIS("home")
    print("  signed in as", gis.users.me.username, flush=True)
    hits = gis.content.search('title:"HF Text Feature Extraction" owner:esri_analytics',
                              item_type="Deep Learning Package") or \
           gis.content.search('title:"HF Text Feature Extraction"',
                              item_type="Deep Learning Package", outside_org=True)
    assert hits, "Living Atlas item not found — check sign-in/subscription"
    TEXT_DLPK = hits[0].download(save_path=os.path.dirname(TEXT_DLPK) or HOME)
    print("  downloaded ->", TEXT_DLPK, flush=True)
    mark(STEP, path=TEXT_DLPK)

## 3 — Embed the masked narratives (the long cell)

At `SAMPLE_N = 400` on CPU, budget **~5-15 min** (your smoke test embedded 60
in ~2 min). The tool prints nothing until it finishes — judge by the clock,
not the silence. If Pro dies mid-run, just re-run this cell after restarting:
it clears its own partial output first.

In [ ]:
STEP = "4_embed"
EMB_FC = os.path.join(GDB, "narratives_emb")

if done(STEP) and arcpy.Exists(EMB_FC):
    print("skipped — already done", flush=True)
else:
    if arcpy.Exists(EMB_FC):
        arcpy.management.Delete(EMB_FC)
        print("  deleted partial output from an earlier run", flush=True)
    t0 = begin(STEP, "~5-15 min at SAMPLE_N=400 on CPU")
    arcpy.geoai.GenerateEmbeddingsUsingAIModels(
        FC, EMB_FC, TEXT_DLPK,
        [["text_field", "details_masked"], ["batch_size", "16"]])
    fields = {f.name: f.type for f in arcpy.ListFields(EMB_FC)}
    assert fields.get("Embedding") == "Blob", "no Blob field named Embedding"
    print(f"embedded {arcpy.management.GetCount(EMB_FC)[0]} narratives "
          f"| {(time.time()-t0)/60:.1f} min", flush=True)
    mark(STEP)

## 4 — Unpack to a numeric matrix (~1-3 min)

In [ ]:
STEP = "5_extract"
FLAT_FC = os.path.join(GDB, "narratives_emb_flat")
X_NPY   = os.path.join(HOME, "X.npy")
PK_CSV  = os.path.join(HOME, "pk_order.csv")

if done(STEP) and os.path.exists(X_NPY):
    X = np.load(X_NPY)
    pk_order = pd.read_csv(PK_CSV)["pk"].to_numpy()
    print(f"skipped — already done (X {X.shape})", flush=True)
else:
    t0 = begin(STEP, "~1-3 min (adds ~384 fields)")
    if arcpy.Exists(FLAT_FC):
        arcpy.management.Delete(FLAT_FC)
    arcpy.geoai.ExtractEmbeddingToFields(EMB_FC, FLAT_FC, "Embedding", "emb")
    emb_cols = [f.name for f in arcpy.ListFields(FLAT_FC)
                if f.name.lower().startswith("emb") and f.type in ("Double", "Single")]
    emb_cols.sort(key=lambda c: int(re.sub(r"\D", "", c) or 0))
    rows, pks = [], []
    with arcpy.da.SearchCursor(FLAT_FC, ["pk"] + emb_cols) as cur:
        for r in cur:
            pks.append(r[0]); rows.append(r[1:])
    X = np.asarray(rows, dtype=np.float32)
    pk_order = np.asarray(pks)
    np.save(X_NPY, X)
    pd.DataFrame({"pk": pk_order}).to_csv(PK_CSV, index=False)
    print(f"X {X.shape} saved | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, dims=int(X.shape[1]))

narr_x = narr.set_index("pk").loc[pk_order].reset_index()
assert len(narr_x) == X.shape[0]

## 5 — Cluster and read the clusters (~1 min)

PCA to 15 components, HDBSCAN (`min_cluster_size=15` at MVP scale; label −1 =
unclustered noise, which is fine), KMeans fallback on older sklearn. Then for
each cluster: top TF-IDF terms and 3 sample narratives — **read them and judge
whether they describe coherent mechanisms.** That judgment is the MVP's main
output.

In [ ]:
STEP = "6_cluster"
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = begin(STEP, "~1 min")
Z = PCA(n_components=15, random_state=SEED).fit_transform(normalize(X))

labels, method = None, None
if HAS_HDBSCAN:
    from sklearn.cluster import HDBSCAN
    labels = HDBSCAN(min_cluster_size=15).fit_predict(Z)
    method = "HDBSCAN(min_cluster_size=15)"
    n_real = len(set(labels)) - (1 if -1 in labels else 0)
    noise = float((labels == -1).mean())
    print(f"HDBSCAN: {n_real} clusters, {noise:.0%} noise", flush=True)
    if n_real < 2 or noise > 0.5:
        print("HDBSCAN mostly noise at this sample size — falling back to KMeans", flush=True)
        labels = None

if labels is None:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    best = None
    for k in range(4, 11):
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z)
        s = silhouette_score(Z, km.labels_)
        print(f"  k={k:2d} silhouette={s:.3f}", flush=True)
        if best is None or s > best[0]:
            best = (s, k, km.labels_)
    labels = best[2]; method = f"KMeans(k={best[1]})"

narr_x["cluster"] = labels
print(method, "| cluster sizes:", flush=True)
print(narr_x["cluster"].value_counts().sort_index().to_string(), flush=True)

vec = TfidfVectorizer(stop_words="english", max_features=4000, ngram_range=(1, 2), min_df=3)
T = vec.fit_transform(narr_x["details_masked"])
terms = np.asarray(vec.get_feature_names_out())
for c in sorted(set(labels)):
    if c == -1:
        continue
    in_c = labels == c
    score = T[in_c].mean(axis=0).A1 - T[~in_c].mean(axis=0).A1
    top = terms[np.argsort(score)[::-1][:10]]
    print("=" * 70, flush=True)
    print(f"CLUSTER {c}  (n={in_c.sum()}, injured {narr_x.loc[in_c,'injured'].mean():.0%})", flush=True)
    print("terms:", ", ".join(top), flush=True)
    for s in narr_x.loc[in_c].sample(min(3, int(in_c.sum())), random_state=SEED)["details_masked"]:
        print("  •", str(s)[:170].replace("\n", " "), flush=True)

narr_x.to_csv(os.path.join(HOME, "narratives_clustered.csv"), index=False, encoding="utf-8-sig")
print(f"{time.time()-t0:.0f}s", flush=True)
mark(STEP, method=method)

## 6 — Validate: injury rate by cluster (~10 s)

Rates with 95% Wilson intervals and a chi-square. Because clustering used
masked text, differing rates point at mechanism, not at riders describing
injuries. Non-overlapping intervals between at least two clusters = the MVP's
quantitative success signal.

In [ ]:
STEP = "7_validate"
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint

t0 = begin(STEP, "~10 s")
clustered = narr_x[narr_x["cluster"] >= 0]
assert len(clustered) > 0 and clustered["cluster"].nunique() >= 2, (
    "fewer than 2 clusters to validate — re-run step 6 (it now falls back to "
    "KMeans when HDBSCAN returns mostly noise)")
rows = []
for c, g in clustered.groupby("cluster"):
    lo, hi = proportion_confint(g["injured"].sum(), len(g), method="wilson")
    rows.append({"cluster": c, "n": len(g), "injury_rate": g["injured"].mean(),
                 "ci_low": lo, "ci_high": hi, "ebike_share": (g["ebike"] == "Yes").mean()})
summary = pd.DataFrame(rows).sort_values("injury_rate", ascending=False)
print(summary.to_string(index=False,
      formatters={c: "{:.1%}".format for c in ["injury_rate", "ci_low", "ci_high", "ebike_share"]}),
      flush=True)

ct = pd.crosstab(narr_x.loc[narr_x["cluster"] >= 0, "cluster"],
                 narr_x.loc[narr_x["cluster"] >= 0, "injured"])
chi2, p, dof, _ = chi2_contingency(ct)
print(f"\nchi-square: chi2={chi2:.1f}, dof={dof}, p={p:.2g} "
      f"| pooled rate {narr_x['injured'].mean():.1%}", flush=True)
summary.to_csv(os.path.join(HOME, "typology_summary.csv"), index=False)
mark(STEP)
status()

## Done — MVP verdict and scaling up

**Judge the MVP on two things:** do the clusters read as coherent mechanisms
(step 5), and do injury rates differ beyond noise (step 6)? Both yes → the
module concept works.

**To scale to the full corpus:** set `SAMPLE_N = None` and point `HOME` at a
*new* folder (outputs are keyed to the folder; mixing sample and full runs in
one folder confuses the resume logic). Full-corpus embedding is the only slow
step (~30-60 min CPU). At full scale, raise `min_cluster_size` to ~30.

**What the full module adds on top of this MVP:** bootstrap stability check on
the clusters, the student-facing naming step (`CLUSTER_NAMES` dict), the map
layer symbolized by typology, the raw-vs-masked leakage exercise, and the
typology-to-odds-ratio regression. All of it reuses this notebook's outputs.